read in the data

In [5]:
from data_loader import load_split

In [6]:
data_dir = "C:\\Users\\nia_4\\SMU\\8sem\\nlp\\project\\data"

X_train, y_train = load_split(data_dir, "media", "train")
X_val, y_val = load_split(data_dir, "media", "valid")
X_test, y_test = load_split(data_dir, "media", "test")

print(len(X_train), len(X_val), len(X_test))

26590 2356 1300


feature extraction

In [44]:
from feature_extraction_stylometric import extract_features


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\nia_4\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nia_4\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\nia_4\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [33]:
def build_feature_matrix(texts):
    return np.array([extract_features(t) for t in texts])

In [36]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_feat = build_feature_matrix(X_train)
X_val_feat = build_feature_matrix(X_val)

X_train_scaled = scaler.fit_transform(X_train_feat)
X_val_scaled = scaler.transform(X_val_feat)

In [37]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model = LogisticRegression(max_iter=2000, solver = "saga", random_state=42, verbose=1)
model.fit(X_train_scaled, y_train)

val_preds = model.predict(X_val_scaled)

print(classification_report(y_val, val_preds))

convergence after 62 epochs took 2 seconds
              precision    recall  f1-score   support

           0       0.45      0.09      0.15      1640
           1       0.70      0.30      0.42       618
           2       0.04      0.64      0.07        98

    accuracy                           0.17      2356
   macro avg       0.40      0.34      0.21      2356
weighted avg       0.50      0.17      0.22      2356



First run of the simplest logistic regression model performed with a weighted average F1 of 0.14 using only basic stylistic features (mean sentence length, mean word length, uppercase words, punctiation, and number of words) like those explored in [Linguistic features based framework for automatic fake news detection (2022)](https://www.sciencedirect.com/science/article/pii/S0360835222004697). To try to help the model capture more information from the text, we added additional features combining stylometry with deeper NLP analysis similar to those used by [Stylometric Fake News Detection Based on NLP with Named Entity Recognition (2023)](https://www.mdpi.com/2079-9292/12/17/3676). Once again running the logitisic regression, we see a poerformance F1 score of 0.21, an improvement with the added POS tag frequencies, function words from [Stylometry: Authorship Identification in Web Forums using NLP](https://www.researchgate.net/publication/359950211_Stylometry_Authorship_Identification_in_Web_Forums_using_Natural_Language_Processing), and readability stats. Furthermore, we tried to imrpove the log reg model with the addition of lightweight features such as punctuation diversity, stopword ratio, and sentence complexity that could further capture subtle stylistic choices without significant preprocessing overhead or overlap. We pulled some of these features from older works like [Stylometry: Authorship Identification in Web Forums using NLP, 2021](https://www.researchgate.net/publication/359950211_Stylometry_Authorship_Identification_in_Web_Forums_using_Natural_Language_Processing), which resulted in a slight increase in the weighted average F1 to be 0.22.

              precision    recall  f1-score   support

           0       0.49      0.10      0.17      1640
           1       0.53      0.27      0.36       618
           2       0.04      0.68      0.07        98

    accuracy                           0.17      2356
   macro avg       0.35      0.35      0.20      2356
weighted avg       0.48      0.17      0.21      2356



In [27]:
import lightgbm as lgb
model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, n_jobs=-1)
model.fit(X_train_feat, y_train)
val_preds = model.predict(X_val_feat)

print(classification_report(y_val, val_preds))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009563 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9435
[LightGBM] [Info] Number of data points in the train set: 26590, number of used features: 37
[LightGBM] [Info] Start training from score -1.098876
[LightGBM] [Info] Start training from score -1.267233
[LightGBM] [Info] Start training from score -0.954136
              precision    recall  f1-score   support

           0       0.49      0.10      0.17      1640
           1       0.53      0.27      0.36       618
           2       0.04      0.68      0.07        98

    accuracy                           0.17      2356
   macro avg       0.35      0.35      0.20      2356
weighted avg       0.48      0.17      0.21      2356



c:\Users\nia_4\SMU\8sem\nlp\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
